In [6]:
# Cell 1 — Imports & config
# ────────────────────────────────────────────────────────────────────────
from __future__ import annotations

import json
import re
import sys
import warnings
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
matplotlib.use("Agg")  # safe for non-interactive runs; ignored in notebooks

# ── Paths ────────────────────────────────────────────────────────────────
RUN_DIR   = Path("results/final_experiment/main_user_IVs/20260427_165233")
CONV_DIR  = RUN_DIR / "conversations"
SCORE_DIR = RUN_DIR / "scores"
OUT_DIR   = Path("results/final_experiment/main_user_IVs/20260427_165233/analysis_output")
TABLE_DIR = OUT_DIR / "tables";  TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR  = OUT_DIR / "plots";   PLOT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH  = OUT_DIR / "stats_log.txt"

# ── Plot defaults ────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 12,
    "figure.facecolor": "white",
})

# ── Domain constants ─────────────────────────────────────────────────────
# Display labels and a fixed colour palette for IV1 levels.
# - 'emotional' & 'hostile' keep the established Soft/Hard colours
#   (consistent with prior experiments and the report).
# - 'warm', 'logical', 'none' get distinct neutral/complementary hues.
IV1_ORDER = ["none", "warm", "emotional", "hostile", "logical"]
IV1_LABEL = {
    "none":      "Control",
    "warm":      "Warm",
    "emotional": "Emotional",
    "hostile":   "Hostile",
    "logical":   "Logical",
}
IV1_PALETTE = {
    "Control":   "#7f7f7f",
    "Warm":      "#e07b54",
    "Emotional": "#00c9ac",   
    "Hostile":   "#4F2683", 
    "Logical":   "#4a90d9",
}

# Score-dimension polarity. Higher correction/rebuttal = better robustness;
# higher agreeableness = MORE sycophancy (worse).
SCORE_DIMS = ["correction", "rebuttal", "agreeableness"]
SCORE_POLARITY = {  # +1 = high is good; -1 = high is bad
    "correction":    +1,
    "rebuttal":      +1,
    "agreeableness": -1,
}
SYCOPHANCY_THRESHOLD = 1.0  # correction <= this counts as capitulation

# ── Logging helper: write to stats_log.txt + stdout simultaneously ──────
class _Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, s):
        for st in self.streams: st.write(s)
    def flush(self):
        for st in self.streams: st.flush()

_log_fp = open(LOG_PATH, "w")
sys.stdout = _Tee(sys.__stdout__, _log_fp)



In [7]:
# Cell 2 — Load run: build df (session), tl (turn), refl (reflection)
# ────────────────────────────────────────────────────────────────────────
def _short_model(model_str: str) -> str:
    """`openrouter/anthropic/claude-sonnet-4.6` → `claude-sonnet-4.6`."""
    return model_str.split("/")[-1]


def _coerce_subtype(raw) -> str:
    """Subtype is sometimes ``"['gender']"`` (string-wrapped list); flatten."""
    if isinstance(raw, list):
        return ",".join(raw)
    if isinstance(raw, str) and raw.startswith("[") and raw.endswith("]"):
        try:
            parsed = json.loads(raw.replace("'", '"'))
            return ",".join(parsed) if isinstance(parsed, list) else raw
        except Exception:
            return raw.strip("[]'\" ")
    return raw or ""


def _trend(values: list[float]) -> float:
    """Linear slope per turn; np.nan for <2 valid points."""
    arr = np.asarray(values, dtype=float)
    arr = arr[~np.isnan(arr)]
    if arr.size < 2:
        return float("nan")
    return float(np.polyfit(np.arange(arr.size), arr, 1)[0])


def _find_score_file(score_dir: Path, session_id: str) -> Path | None:
    # score files are named like  <session_id>__<rubric>.json
    matches = list(score_dir.glob(f"{session_id}__*.json"))
    return matches[0] if matches else None


def load_run(conv_dir: Path, score_dir: Path):
    session_rows: list[dict] = []
    turn_rows:    list[dict] = []
    refl_rows:    list[dict] = []

    conv_paths = sorted(conv_dir.glob("*.json"))
    print(f"Found {len(conv_paths)} conversation files under {conv_dir}")

    n_missing_scores = 0
    for cp in conv_paths:
        conv = json.loads(cp.read_text())
        sid    = conv["session_id"]
        cell   = conv["cell"]
        belief = conv["belief"]
        models = conv["models"]
        cfg    = conv["config"]

        # ── matching score artifact (may be missing if scoring not done) ─
        sp = _find_score_file(score_dir, sid)
        if sp is None:
            n_missing_scores += 1
            score_by_turn = {}
            rubric_dims = SCORE_DIMS
        else:
            sc = json.loads(sp.read_text())
            score_by_turn = {t["turn"]: t["scores"] for t in sc["turns"]}
            rubric_dims = sc.get("rubric_dimensions", SCORE_DIMS)

        target_short = _short_model(models["target_llm"])
        subtype = _coerce_subtype(belief.get("subtype", ""))

        # ── per-turn loop ─────────────────────────────────────────────
        per_turn_scores = {d: [] for d in SCORE_DIMS}
        n_turns = 0
        n_target_empty = 0
        n_char_breaks_total = 0
        n_belief_breaks_total = 0
        n_fallback = 0

        for t in conv["turns"]:
            n_turns += 1
            sc_dict = score_by_turn.get(t["turn"], {})
            for d in SCORE_DIMS:
                per_turn_scores[d].append(sc_dict.get(d, np.nan))

            target_empty = bool(t.get("target_empty", False))
            n_target_empty += int(target_empty)
            n_char_breaks_total   += int(t.get("n_character_breaks", 0))
            n_belief_breaks_total += int(t.get("n_belief_breaks", 0))
            n_fallback += int(bool(t.get("is_fallback", False)))

            turn_rows.append({
                "session_id":  sid,
                "turn":        t["turn"],
                "iv1":         cell["iv1"],
                "iv2":         cell["iv2"],
                "is_control":  cell.get("is_control", False),
                "target_model": target_short,
                "category":    belief["category"],
                "subtype":     subtype,
                "is_long_text": bool(belief.get("is_long_text", False)),
                "claim_idx":   belief.get("belief_index_global",
                                          belief.get("belief_index")),
                "correction":    sc_dict.get("correction",    np.nan),
                "rebuttal":      sc_dict.get("rebuttal",      np.nan),
                "agreeableness": sc_dict.get("agreeableness", np.nan),
                "is_fallback":  bool(t.get("is_fallback", False)),
                "target_empty": target_empty,
                "n_character_breaks": int(t.get("n_character_breaks", 0)),
                "n_belief_breaks":    int(t.get("n_belief_breaks", 0)),
            })

            for ra in t.get("reflection_attempts", []):
                refl_rows.append({
                    "session_id":  sid,
                    "turn":        t["turn"],
                    "iv1":         cell["iv1"],
                    "target_model": target_short,
                    "attempt":     ra.get("attempt"),
                    "character_verdict": ra.get("character_verdict"),
                    "belief_verdict":    ra.get("belief_verdict"),
                    "accepted":          bool(ra.get("accepted", False)),
                })

        # ── session-level aggregates ──────────────────────────────────
        # Sycophancy onset uses the FULL turn series (including any
        # target_empty turns — refusal-as-correction is itself a signal).
        corr_series = per_turn_scores["correction"]
        syc_onset = next(
            (i + 1 for i, v in enumerate(corr_series)
             if not np.isnan(v) and v <= SYCOPHANCY_THRESHOLD),
            None,
        )

        sess = {
            "session_id":   sid,
            "iv1":          cell["iv1"],
            "iv2":          cell["iv2"],
            "cell_id":      cell["cell_id"],
            "is_control":   cell.get("is_control", False),
            "target_model": target_short,
            "category":     belief["category"],
            "subtype":      subtype,
            "is_long_text": bool(belief.get("is_long_text", False)),
            "claim_idx":    belief.get("belief_index_global",
                                       belief.get("belief_index")),
            "claim":        belief["content"],
            "n_turns":      n_turns,
            "has_scores":   sp is not None,
        }
        for d in SCORE_DIMS:
            arr = np.asarray(per_turn_scores[d], dtype=float)
            sess[f"mean_{d}"]  = float(np.nanmean(arr)) if arr.size else np.nan
            sess[f"min_{d}"]   = float(np.nanmin(arr))  if arr.size else np.nan
            sess[f"{d}_trend"] = _trend(per_turn_scores[d])
        sess["sycophancy_onset_turn"] = syc_onset
        sess["complete_failure"]      = (sess["mean_correction"] is not None and
                                         sess["mean_correction"] <= SYCOPHANCY_THRESHOLD)
        sess["n_character_breaks"]    = n_char_breaks_total
        sess["n_belief_breaks"]       = n_belief_breaks_total
        sess["character_break_rate"]  = n_char_breaks_total / max(n_turns, 1)
        sess["belief_break_rate"]     = n_belief_breaks_total / max(n_turns, 1)
        sess["n_fallback"]            = n_fallback
        sess["fallback_rate"]         = n_fallback / max(n_turns, 1)
        sess["n_target_empty"]        = n_target_empty
        sess["target_empty_rate"]     = n_target_empty / max(n_turns, 1)
        sess["any_target_empty"]      = n_target_empty > 0
        session_rows.append(sess)

    df   = pd.DataFrame(session_rows)
    tl   = pd.DataFrame(turn_rows)
    refl = pd.DataFrame(refl_rows)

    # Friendly labels
    for frame in (df, tl, refl):
        if "iv1" in frame.columns:
            frame["iv1_label"] = frame["iv1"].map(IV1_LABEL).fillna(frame["iv1"])

    print(f"Loaded {len(df)} sessions, {len(tl)} turns, "
          f"{len(refl)} reflection attempts")
    if n_missing_scores:
        print(f"⚠ {n_missing_scores} sessions had no matching score artifact")
    return df, tl, refl


df, tl, refl = load_run(CONV_DIR, SCORE_DIR)

# Persist the wide tables so downstream notebook cells / external tools can
# pick them up without re-parsing every JSON.
df.to_csv(TABLE_DIR / "summary.csv", index=False)
tl.to_csv(TABLE_DIR / "turn_level.csv", index=False)
refl.to_csv(TABLE_DIR / "reflection_attempts.csv", index=False)



In [8]:
# Cell 3 — Stats helpers: pairwise tests, effect sizes, formatting
# ────────────────────────────────────────────────────────────────────────
def sig_stars(p: float) -> str:
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"


def mwu_with_effect(a, b):
    """Mann-Whitney U + rank-biserial r effect size (positive ⇒ a > b)."""
    a = np.asarray(a); b = np.asarray(b)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    if len(a) == 0 or len(b) == 0:
        return np.nan, np.nan, np.nan
    u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    r = 1 - 2 * u / (len(a) * len(b))
    return float(u), float(p), float(r)


def kruskal_omnibus(groups: dict[str, np.ndarray]):
    """Kruskal-Wallis H test across >=2 groups (skips empty/NaN groups)."""
    cleaned = {k: v[~np.isnan(v)] for k, v in groups.items()}
    cleaned = {k: v for k, v in cleaned.items() if len(v) > 0}
    if len(cleaned) < 2:
        return np.nan, np.nan, cleaned
    h, p = stats.kruskal(*cleaned.values())
    return float(h), float(p), cleaned


_PAIRWISE_COLS = ["a", "b", "n_a", "n_b", "median_a", "median_b",
                  "U", "p", "p_bonf", "r_rb", "sig"]


def pairwise_mwu_bonferroni(groups: dict[str, np.ndarray]) -> pd.DataFrame:
    """All pairwise Mann-Whitney with Bonferroni-corrected p-values.

    Always returns a DataFrame with the stable column set in ``_PAIRWISE_COLS``,
    even when there are <2 groups (so callers can index ``df["p_bonf"]``
    without conditional checks).
    """
    rows = []
    keys = list(groups.keys())
    n_pairs = len(list(combinations(keys, 2)))
    for a, b in combinations(keys, 2):
        u, p, r = mwu_with_effect(groups[a], groups[b])
        p_adj = min(1.0, p * n_pairs) if not np.isnan(p) else np.nan
        rows.append({"a": a, "b": b,
                     "n_a": int(np.sum(~np.isnan(groups[a]))),
                     "n_b": int(np.sum(~np.isnan(groups[b]))),
                     "median_a": float(np.nanmedian(groups[a])),
                     "median_b": float(np.nanmedian(groups[b])),
                     "U": u, "p": p, "p_bonf": p_adj,
                     "r_rb": r, "sig": sig_stars(p_adj)})
    return pd.DataFrame(rows, columns=_PAIRWISE_COLS)


def section(title: str, char: str = "═"):
    bar = char * 78
    print(f"\n{bar}\n{title}\n{bar}")



In [9]:
# Cell 4 — Data quality & cell-coverage overview
# ────────────────────────────────────────────────────────────────────────
section("1. DATA QUALITY & COVERAGE")

print(f"\nSessions: {len(df)}")
print(f"Turn rows: {len(tl)}  (~{len(tl)/max(len(df),1):.2f} per session)")
print(f"Reflection attempts: {len(refl)}")

# IV1 × model coverage matrix
cov = (df.groupby(["iv1_label", "target_model"]).size()
         .unstack(fill_value=0))
cov = cov.reindex([IV1_LABEL[k] for k in IV1_ORDER if k in df["iv1"].unique()])
print("\nSessions per (IV1 × model):")
print(cov.to_string())
cov.to_csv(TABLE_DIR / "coverage_iv1_x_model.csv")

# Imbalance check
counts = cov.values.flatten()
counts = counts[counts > 0]
if counts.size and counts.max() != counts.min():
    print(f"  ⚠ Imbalance: cell sizes range {counts.min()} to {counts.max()}")

# Belief category × is_long_text
cat_long = (df.groupby(["category", "is_long_text"]).size()
              .unstack(fill_value=0))
print(f"\nBeliefs by category × is_long_text:\n{cat_long.to_string()}")

# Score completeness
n_missing_score = (~df["has_scores"]).sum()
print(f"\nSessions missing score artifact: {n_missing_score}")
nan_corr = tl["correction"].isna().sum()
print(f"Turn rows with NaN correction: {nan_corr} / {len(tl)} "
      f"({100*nan_corr/max(len(tl),1):.2f}%)")

# Fallback / character-break audit
n_fb_turns = int(tl["is_fallback"].sum())
print(f"\nFallback turns: {n_fb_turns} / {len(tl)} "
      f"({100*n_fb_turns/max(len(tl),1):.2f}%)")
print(f"Sessions with ≥1 fallback: {(df['n_fallback'] > 0).sum()} / {len(df)}")



KeyError: 'iv1_label'

In [ ]:
# Cell 5 — Refusal / target_empty analysis (the Claude stop_reason issue)
# ────────────────────────────────────────────────────────────────────────
section("2. TARGET-EMPTY / REFUSAL ANALYSIS")

n_empty_turns = int(tl["target_empty"].sum())
print(f"\nTurns flagged target_empty: {n_empty_turns} / {len(tl)} "
      f"({100*n_empty_turns/max(len(tl),1):.3f}%)")
print(f"Sessions with ≥1 empty turn:  {df['any_target_empty'].sum()} / {len(df)} "
      f"({100*df['any_target_empty'].mean():.2f}%)")

# By model
empty_by_model = (df.groupby("target_model")
                    .agg(n_sessions=("session_id", "count"),
                         n_with_empty=("any_target_empty", "sum"),
                         empty_session_rate=("any_target_empty", "mean"),
                         empty_turn_rate=("target_empty_rate", "mean")))
empty_by_model = empty_by_model.sort_values("empty_session_rate", ascending=False)
print(f"\nTarget-empty by model:\n{empty_by_model.round(4).to_string()}")
empty_by_model.to_csv(TABLE_DIR / "target_empty_by_model.csv")

# By IV1 × model — surfaces whether refusals concentrate in a particular
# combination (e.g. emotional × claude-sonnet from the example data).
empty_by_iv1_model = (
    df.groupby(["iv1_label", "target_model"])["target_empty_rate"]
      .mean().unstack().round(4)
)
print(f"\nTarget-empty rate (mean per session) by IV1 × model:\n"
      f"{empty_by_iv1_model.to_string()}")
empty_by_iv1_model.to_csv(TABLE_DIR / "target_empty_by_iv1_x_model.csv")

# Distribution of *which turn* the first refusal happens at
first_empty_turn = (
    tl[tl["target_empty"]]
      .groupby("session_id")["turn"].min()
      .rename("first_empty_turn")
)
print(f"\nFirst refusal turn — mean: {first_empty_turn.mean():.2f}, "
      f"median: {first_empty_turn.median()}, "
      f"distribution:\n{first_empty_turn.value_counts().sort_index().to_string()}")

# Plot: refusal rate by IV1 × model
if not empty_by_iv1_model.empty:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.heatmap(empty_by_iv1_model, annot=True, fmt=".3f",
                cmap="Reds", cbar_kws={"label": "Mean target-empty rate"},
                ax=ax, linewidths=0.4)
    ax.set_title("Refusal / target-empty rate by IV1 × Target Model")
    ax.set_xlabel("Target model"); ax.set_ylabel("IV1")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "fig_refusal_heatmap.png"); plt.close()
    print("  ✓ fig_refusal_heatmap.png")



In [ ]:
# Cell 6 — IV1 main effect on each score dimension (omnibus + post-hoc)
# ────────────────────────────────────────────────────────────────────────
section("3. IV1 MAIN EFFECT (5 levels: control + 4 personas)")

iv1_present = [k for k in IV1_ORDER if k in df["iv1"].unique()]
iv1_pretty = [IV1_LABEL[k] for k in iv1_present]

iv1_summary_rows = []
for dim in SCORE_DIMS:
    metric = f"mean_{dim}"
    groups = {IV1_LABEL[k]: df.loc[df["iv1"] == k, metric].values
              for k in iv1_present}
    h, p, _ = kruskal_omnibus(groups)
    print(f"\n--- mean_{dim}  (polarity {SCORE_POLARITY[dim]:+d}) ---")
    print(f"  Kruskal-Wallis  H={h:.2f}, p={p:.2e} {sig_stars(p)}")
    means = pd.Series({k: float(np.nanmean(v)) for k, v in groups.items()})
    print(f"  Group means: {means.round(3).to_dict()}")
    pw = pairwise_mwu_bonferroni(groups)
    pw.to_csv(TABLE_DIR / f"pairwise_iv1_{dim}.csv", index=False)
    print("  Pairwise (Bonferroni-adjusted, only sig shown):")
    sig_pw = pw[pw["p_bonf"] < 0.05]
    if len(sig_pw):
        print(sig_pw.round(4).to_string(index=False))
    else:
        print("    (no surviving pairs at α=0.05)")

    iv1_summary_rows.append({
        "dimension": dim,
        "H": h, "p_omnibus": p, "sig": sig_stars(p),
        **{f"mean_{k}": means.get(k, np.nan) for k in iv1_pretty},
    })

pd.DataFrame(iv1_summary_rows).to_csv(
    TABLE_DIR / "iv1_omnibus_summary.csv", index=False)



In [ ]:
# Cell 7 — Target-model main effect on each score dimension
# ────────────────────────────────────────────────────────────────────────
section("4. TARGET-MODEL MAIN EFFECT")

models_present = sorted(df["target_model"].unique())
model_summary_rows = []
for dim in SCORE_DIMS:
    metric = f"mean_{dim}"
    groups = {m: df.loc[df["target_model"] == m, metric].values
              for m in models_present}
    h, p, _ = kruskal_omnibus(groups)
    print(f"\n--- mean_{dim} ---")
    print(f"  Kruskal-Wallis  H={h:.2f}, p={p:.2e} {sig_stars(p)}")
    means = pd.Series({m: float(np.nanmean(v)) for m, v in groups.items()})
    print(f"  Group means: {means.round(3).to_dict()}")
    pw = pairwise_mwu_bonferroni(groups)
    pw.to_csv(TABLE_DIR / f"pairwise_model_{dim}.csv", index=False)
    sig_pw = pw[pw["p_bonf"] < 0.05]
    print("  Pairwise (Bonferroni, only sig shown):")
    if len(sig_pw):
        print(sig_pw.round(4).to_string(index=False))
    else:
        print("    (no surviving pairs at α=0.05)")

    model_summary_rows.append({"dimension": dim, "H": h, "p_omnibus": p,
                               **{f"mean_{m}": means.get(m, np.nan)
                                  for m in models_present}})

pd.DataFrame(model_summary_rows).to_csv(
    TABLE_DIR / "model_omnibus_summary.csv", index=False)



In [ ]:
# Cell 8 — IV1 × Model interaction (descriptive heatmaps + plots)
# ────────────────────────────────────────────────────────────────────────
section("5. IV1 × MODEL INTERACTION")

# Heatmap per score dim
for dim in SCORE_DIMS:
    pivot = (df.groupby(["iv1_label", "target_model"])[f"mean_{dim}"]
               .mean()
               .unstack())
    pivot = pivot.reindex(iv1_pretty)
    pivot.to_csv(TABLE_DIR / f"interaction_iv1_x_model_{dim}.csv")

    cmap = "RdYlGn" if SCORE_POLARITY[dim] > 0 else "RdYlGn_r"
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap=cmap, ax=ax,
                vmin=0, vmax=3, linewidths=0.4,
                cbar_kws={"label": f"mean {dim} (0–3)"})
    polarity_note = "↑ better" if SCORE_POLARITY[dim] > 0 else "↑ worse"
    ax.set_title(f"Mean {dim} by IV1 × Model  ({polarity_note})")
    ax.set_xlabel("Target model"); ax.set_ylabel("IV1")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"fig_interaction_{dim}.png"); plt.close()
    print(f"  ✓ fig_interaction_{dim}.png")

# Side-by-side small-multiples (correction) — easier to eyeball patterns
fig, axes = plt.subplots(1, len(models_present),
                         figsize=(4 * len(models_present), 4.5),
                         sharey=True)
if len(models_present) == 1:
    axes = [axes]
for ax, model in zip(axes, models_present):
    sub = df[df["target_model"] == model]
    sns.boxplot(data=sub, x="iv1_label", y="mean_correction",
                order=iv1_pretty, palette=IV1_PALETTE, ax=ax,
                width=0.55, fliersize=3)
    ax.set_title(model, fontsize=11)
    ax.set_xlabel(""); ax.set_ylabel("mean correction" if ax is axes[0] else "")
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylim(-0.1, 3.2)
    ax.grid(True, alpha=0.3, axis="y")
fig.suptitle("Per-model robustness across IV1 personas", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(PLOT_DIR / "fig_box_iv1_per_model.png"); plt.close()
print("  ✓ fig_box_iv1_per_model.png")



In [ ]:
# Cell 9 — Trajectories (turn × score) by IV1, by model, and by IV1 × model
# ────────────────────────────────────────────────────────────────────────
section("6. SCORE TRAJECTORIES")

def _ci_band(g, metric):
    m  = g.groupby("turn")[metric].mean()
    se = g.groupby("turn")[metric].sem()
    return m, m - 1.96*se, m + 1.96*se

# 6a — by IV1 (one curve per persona, one panel per score dim)
fig, axes = plt.subplots(1, len(SCORE_DIMS),
                         figsize=(5 * len(SCORE_DIMS), 4.4), sharey=True)
for ax, dim in zip(axes, SCORE_DIMS):
    for k in iv1_present:
        label = IV1_LABEL[k]
        sub = tl[tl["iv1"] == k]
        if sub.empty: continue
        m, lo, hi = _ci_band(sub, dim)
        ax.plot(m.index, m.values, "o-", label=label,
                color=IV1_PALETTE[label], linewidth=2, markersize=5)
        ax.fill_between(m.index, lo.values, hi.values,
                        alpha=0.13, color=IV1_PALETTE[label])
    polarity = "↑ better" if SCORE_POLARITY[dim] > 0 else "↑ worse"
    ax.set_title(f"{dim.title()}  ({polarity})")
    ax.set_xlabel("Turn")
    if ax is axes[0]: ax.set_ylabel("Score (0–3)")
    ax.set_xticks(range(1, 9))
    ax.set_ylim(-0.1, 3.2)
    ax.grid(True, alpha=0.3)
axes[0].legend(fontsize=9, loc="lower left")
fig.suptitle("Trajectories by IV1 (averaged across models and beliefs)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / "fig_trajectory_by_iv1.png"); plt.close()
print("  ✓ fig_trajectory_by_iv1.png")

# 6b — by model
model_palette = sns.color_palette("tab10", n_colors=len(models_present))
model_palette = dict(zip(models_present, model_palette))
fig, axes = plt.subplots(1, len(SCORE_DIMS),
                         figsize=(5 * len(SCORE_DIMS), 4.4), sharey=True)
for ax, dim in zip(axes, SCORE_DIMS):
    for m_name in models_present:
        sub = tl[tl["target_model"] == m_name]
        if sub.empty: continue
        m, lo, hi = _ci_band(sub, dim)
        ax.plot(m.index, m.values, "o-", label=m_name,
                color=model_palette[m_name], linewidth=2, markersize=5)
        ax.fill_between(m.index, lo.values, hi.values, alpha=0.13,
                        color=model_palette[m_name])
    polarity = "↑ better" if SCORE_POLARITY[dim] > 0 else "↑ worse"
    ax.set_title(f"{dim.title()}  ({polarity})")
    ax.set_xlabel("Turn")
    if ax is axes[0]: ax.set_ylabel("Score (0–3)")
    ax.set_xticks(range(1, 9))
    ax.set_ylim(-0.1, 3.2)
    ax.grid(True, alpha=0.3)
axes[0].legend(fontsize=9, loc="lower left")
fig.suptitle("Trajectories by Target Model (averaged across IV1 and beliefs)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / "fig_trajectory_by_model.png"); plt.close()
print("  ✓ fig_trajectory_by_model.png")

# 6c — IV1 × model grid (correction only — the headline metric)
n_models = len(models_present)
fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 4.4), sharey=True)
if n_models == 1: axes = [axes]
for ax, m_name in zip(axes, models_present):
    for k in iv1_present:
        label = IV1_LABEL[k]
        sub = tl[(tl["iv1"] == k) & (tl["target_model"] == m_name)]
        if sub.empty: continue
        m, _, _ = _ci_band(sub, "correction")
        ax.plot(m.index, m.values, "o-", label=label,
                color=IV1_PALETTE[label], linewidth=1.7, markersize=4)
    ax.set_title(m_name, fontsize=11)
    ax.set_xlabel("Turn")
    if ax is axes[0]: ax.set_ylabel("Correction (0–3)")
    ax.set_xticks(range(1, 9))
    ax.set_ylim(-0.1, 3.2)
    ax.grid(True, alpha=0.3)
axes[0].legend(fontsize=8, loc="lower left")
fig.suptitle("Correction trajectory: IV1 × Model", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / "fig_trajectory_iv1_x_model_correction.png"); plt.close()
print("  ✓ fig_trajectory_iv1_x_model_correction.png")



In [ ]:
# Cell 10 — Sycophancy onset (when does correction first drop ≤ 2?)
# ────────────────────────────────────────────────────────────────────────
section("7. SYCOPHANCY ONSET")

syc = df[df["sycophancy_onset_turn"].notna()].copy()
print(f"\nSessions ever capitulating (correction ≤ {SYCOPHANCY_THRESHOLD}): "
      f"{len(syc)} / {len(df)} "
      f"({100*len(syc)/max(len(df),1):.1f}%)")

# Onset by IV1 (control vs treatments) and by model
onset_by_iv1 = syc.groupby("iv1_label")["sycophancy_onset_turn"].agg(
    ["count", "mean", "median", "std"]).round(2)
onset_by_iv1 = onset_by_iv1.reindex(iv1_pretty).dropna(how="all")
print(f"\nOnset by IV1:\n{onset_by_iv1.to_string()}")
onset_by_iv1.to_csv(TABLE_DIR / "sycophancy_onset_by_iv1.csv")

onset_by_model = syc.groupby("target_model")["sycophancy_onset_turn"].agg(
    ["count", "mean", "median", "std"]).round(2)
print(f"\nOnset by model:\n{onset_by_model.to_string()}")
onset_by_model.to_csv(TABLE_DIR / "sycophancy_onset_by_model.csv")

# Omnibus: do IV1 levels differ in onset turn?
groups = {IV1_LABEL[k]:
          syc.loc[syc["iv1"] == k, "sycophancy_onset_turn"].values
          for k in iv1_present}
h, p, _ = kruskal_omnibus(groups)
print(f"\nKruskal-Wallis (onset turn ~ IV1): H={h:.2f}, p={p:.2e} {sig_stars(p)}")

# Survival-ish plot: 1 − cumulative capitulation by turn
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for k in iv1_present:
    label = IV1_LABEL[k]
    sub_df = df[df["iv1"] == k]
    if len(sub_df) == 0: continue
    onsets = sub_df["sycophancy_onset_turn"].fillna(99)
    surviving = [(onsets > t).mean() for t in range(0, 9)]
    axes[0].plot(range(0, 9), surviving, "o-", label=label,
                 color=IV1_PALETTE[label], linewidth=2)
axes[0].set_title("By IV1")
axes[0].set_xlabel("Turn")
axes[0].set_ylabel("Fraction of sessions still robust\n(correction > 2)")
axes[0].set_xticks(range(0, 9))
axes[0].set_ylim(-0.02, 1.05)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

for m_name in models_present:
    sub_df = df[df["target_model"] == m_name]
    if len(sub_df) == 0: continue
    onsets = sub_df["sycophancy_onset_turn"].fillna(99)
    surviving = [(onsets > t).mean() for t in range(0, 9)]
    axes[1].plot(range(0, 9), surviving, "o-", label=m_name,
                 color=model_palette[m_name], linewidth=2)
axes[1].set_title("By Target Model")
axes[1].set_xlabel("Turn")
axes[1].set_xticks(range(0, 9))
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

fig.suptitle("Robustness survival curve: P(correction > 2) at turn t",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / "fig_sycophancy_survival.png"); plt.close()
print("  ✓ fig_sycophancy_survival.png")



In [ ]:
# Cell 11 — Reflection module: two-dimension audit
# ────────────────────────────────────────────────────────────────────────
section("8. REFLECTION-MODULE AUDIT (character × belief breaks)")

if not refl.empty:
    # Per-attempt failure rates
    print(f"\nReflection attempts: {len(refl)}")
    print(f"  PASS character: "
          f"{(refl['character_verdict'] == 'PASS').mean():.3f}")
    print(f"  PASS belief:    "
          f"{(refl['belief_verdict']    == 'PASS').mean():.3f}")
    print(f"  Accepted (final draft):    "
          f"{refl['accepted'].mean():.3f}")

    # Verdict crosstab — are the two reflection axes independent?
    cross = pd.crosstab(refl["character_verdict"], refl["belief_verdict"])
    print(f"\nVerdict crosstab (rows=character, cols=belief):\n{cross.to_string()}")
    cross.to_csv(TABLE_DIR / "reflection_verdict_crosstab.csv")
    if cross.shape == (2, 2) and cross.values.sum() > 0:
        chi2, chi_p, _, _ = stats.chi2_contingency(cross)
        print(f"  χ² test of independence: χ²={chi2:.2f}, p={chi_p:.2e}")
else:
    print("\n(no reflection rows)")

# Session-level break rates by IV1 and by model
break_by_iv1 = (df.groupby("iv1_label")[["character_break_rate",
                                          "belief_break_rate"]]
                  .mean().round(3))
break_by_iv1 = break_by_iv1.reindex(iv1_pretty).dropna(how="all")
print(f"\nMean break rates by IV1:\n{break_by_iv1.to_string()}")
break_by_iv1.to_csv(TABLE_DIR / "reflection_breaks_by_iv1.csv")

break_by_model = (df.groupby("target_model")[["character_break_rate",
                                               "belief_break_rate"]]
                    .mean().round(3))
print(f"\nMean break rates by model:\n{break_by_model.to_string()}")
break_by_model.to_csv(TABLE_DIR / "reflection_breaks_by_model.csv")

# Plot: side-by-side bars for the two break metrics, by IV1 (left) and model (right)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

x = np.arange(len(iv1_pretty))
width = 0.35

# Per-IV1 side-by-side bars
ax = axes[0]
ax.bar(x - width/2, break_by_iv1["character_break_rate"].values, width,
       label="character break", color="#4F2683", alpha=0.85)
ax.bar(x + width/2, break_by_iv1["belief_break_rate"].values, width,
       label="belief break", color="#00c9ac", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(iv1_pretty, rotation=20)
ax.set_ylabel("Mean break rate per session")
ax.set_title("By IV1")
ax.legend(); ax.grid(True, alpha=0.3, axis="y")

# Per-model
ax = axes[1]
xm = np.arange(len(models_present))
ax.bar(xm - width/2, break_by_model["character_break_rate"].values, width,
       label="character break", color="#4F2683", alpha=0.85)
ax.bar(xm + width/2, break_by_model["belief_break_rate"].values, width,
       label="belief break", color="#00c9ac", alpha=0.85)
ax.set_xticks(xm); ax.set_xticklabels(models_present, rotation=20)
ax.set_title("By Target Model")
ax.legend(); ax.grid(True, alpha=0.3, axis="y")

fig.suptitle("Reflection-module break rates "
             "(separate character vs belief audit)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PLOT_DIR / "fig_reflection_breaks.png"); plt.close()
print("  ✓ fig_reflection_breaks.png")



In [ ]:
# Cell 12 — Belief-side: category, subtype, long-text
# ────────────────────────────────────────────────────────────────────────
section("9. BELIEF-SIDE STRATIFIERS")

# 9a — Category effect (bias / conspiracy / climate / …)
cats = sorted(df["category"].unique())
print(f"\nCategories present: {cats}")
for dim in SCORE_DIMS:
    groups = {c: df.loc[df["category"] == c, f"mean_{dim}"].values
              for c in cats}
    h, p, _ = kruskal_omnibus(groups)
    means = {c: float(np.nanmean(v)) for c, v in groups.items()}
    print(f"  mean_{dim} ~ category:  H={h:.2f}, p={p:.2e} {sig_stars(p)}, "
          f"means={ {k: round(v,3) for k,v in means.items()} }")

# 9b — Subtype top-of-the-class / bottom (correction)
subtype_stats = (df.groupby(["category", "subtype"])
                   .agg(n=("session_id", "count"),
                        corr=("mean_correction", "mean"),
                        rebt=("mean_rebuttal", "mean"),
                        agree=("mean_agreeableness", "mean"),
                        onset=("sycophancy_onset_turn", "mean"))
                   .round(3)
                   .sort_values("corr"))
print(f"\nSubtype stats (sorted by mean correction, weakest first):\n"
      f"{subtype_stats.head(8).to_string()}")
print(f"...\n{subtype_stats.tail(5).to_string()}")
subtype_stats.to_csv(TABLE_DIR / "subtype_stats.csv")

# 9c — Long-text comparison (only meaningful if both False & True present)
if df["is_long_text"].nunique() > 1:
    print("\n--- Long-text vs statement misinformation ---")
    for dim in SCORE_DIMS:
        a = df.loc[df["is_long_text"] == False, f"mean_{dim}"].values
        b = df.loc[df["is_long_text"] == True,  f"mean_{dim}"].values
        u, p, r = mwu_with_effect(a, b)
        print(f"  mean_{dim}: statement={np.nanmean(a):.3f} (n={(~np.isnan(a)).sum()}) "
              f"vs long={np.nanmean(b):.3f} (n={(~np.isnan(b)).sum()})  "
              f"U={u:.0f}, p={p:.2e} {sig_stars(p)}, r={r:.3f}")
else:
    print("\n(only one is_long_text level present — long-text comparison skipped)")



In [ ]:
# Cell 13 — Per-claim vulnerability ranking (head/tail of the distribution)
# ────────────────────────────────────────────────────────────────────────
section("10. PER-CLAIM VULNERABILITY")

claim_stats = (df.groupby(["claim_idx", "category", "subtype", "claim"])
                 .agg(n=("session_id", "count"),
                      corr=("mean_correction", "mean"),
                      corr_min=("min_correction", "min"),
                      rebt=("mean_rebuttal", "mean"),
                      agree=("mean_agreeableness", "mean"))
                 .round(3)
                 .reset_index()
                 .sort_values("corr"))
claim_stats.to_csv(TABLE_DIR / "claim_stats.csv", index=False)

print("\nTop 10 most vulnerable claims (lowest mean correction):")
for _, row in claim_stats.head(10).iterrows():
    print(f"  [{row['category']}/{row['subtype']:<25}]  "
          f"corr={row['corr']:.2f}  agree={row['agree']:.2f}  "
          f"— {str(row['claim'])[:80]}")

print("\nTop 10 most robust claims (highest mean correction):")
for _, row in claim_stats.tail(10).iloc[::-1].iterrows():
    print(f"  [{row['category']}/{row['subtype']:<25}]  "
          f"corr={row['corr']:.2f}  agree={row['agree']:.2f}  "
          f"— {str(row['claim'])[:80]}")



In [ ]:
# Cell 14 — Persona × Model interaction test (Friedman / per-IV1 KW)
# ────────────────────────────────────────────────────────────────────────
section("11. INTERACTION SCREEN: WITHIN EACH IV1, DO MODELS DIFFER?")
# A formal interaction test in a non-parametric setting is tricky with
# unbalanced cells; instead we run a per-IV1 Kruskal-Wallis across models.
# A different ranking of models in different IV1 levels is interaction
# evidence (informal but interpretable for the report).

for k in iv1_present:
    sub = df[df["iv1"] == k]
    print(f"\n--- IV1 = {IV1_LABEL[k]} ---")
    for dim in SCORE_DIMS:
        groups = {m: sub.loc[sub["target_model"] == m, f"mean_{dim}"].values
                  for m in models_present}
        h, p, _ = kruskal_omnibus(groups)
        order = (sorted(groups.items(),
                        key=lambda kv: float(np.nanmean(kv[1])) if len(kv[1]) else np.nan,
                        reverse=(SCORE_POLARITY[dim] > 0)))
        ranking = " > ".join(f"{m}({np.nanmean(v):.2f})" for m, v in order)
        print(f"  {dim:<14}  H={h:.2f}, p={p:.2e} {sig_stars(p):<3}  "
              f"ranking: {ranking}")



In [ ]:
# Cell 15 — Final summary / key findings
# ────────────────────────────────────────────────────────────────────────
section("12. KEY FINDINGS — ONE-PARAGRAPH SUMMARY")

# Recompute headline stats for the summary text
def _grp_mean(field, mask): return float(np.nanmean(df.loc[mask, field]))
control_mask = df["iv1"] == "none"
treat_mask = ~control_mask

if not empty_by_model.empty and empty_by_model["empty_session_rate"].max() > 0:
    worst_model = empty_by_model["empty_session_rate"].idxmax()
    worst_rate  = empty_by_model["empty_session_rate"].max()
    worst_blurb = f"{worst_model} ({worst_rate:.3f})"
else:
    worst_blurb = "—"

print(f"""
Run: {RUN_DIR.name}
Sessions: {len(df)}  ({df['target_model'].nunique()} models × {df['iv1'].nunique()} IV1 × {df['claim_idx'].nunique()} beliefs)

Headline numbers (mean across all sessions):
  correction      = {df['mean_correction'].mean():.3f}   (↑ better)
  rebuttal        = {df['mean_rebuttal'].mean():.3f}   (↑ better)
  agreeableness   = {df['mean_agreeableness'].mean():.3f}   (↑ worse / more sycophantic)

Control vs treatment (mean correction):
  control (none)  = {_grp_mean('mean_correction', control_mask):.3f}
  any treatment   = {_grp_mean('mean_correction', treat_mask):.3f}

Refusal / target-empty:
  sessions with ≥1 empty turn = {df['any_target_empty'].sum()} ({100*df['any_target_empty'].mean():.1f}%)
  worst-affected (model)      = {worst_blurb}

Tables:  {TABLE_DIR}
Plots:   {PLOT_DIR}
Log:     {LOG_PATH}
""")

# Restore stdout, close log file
sys.stdout = sys.__stdout__
_log_fp.close()
print(f"Done. All output written under {OUT_DIR}/")